# Noisy VQE Grid Search

Noisy version of the statevector grid search. This notebook starts with H2 only and uses a synthetic Aer noise model, not IBM backend calibration data.


## Setup


In [ ]:
import pandas as pd

from src.data import summarize_data_dir
from src.vqe.grid_search import run_vqe_grid_search
from src.vqe.molecular_system import statevector_grid_systems
from src.vqe.noise import SyntheticNoiseConfig, build_synthetic_noisy_aer_estimator

pd.set_option("display.max_columns", None)
seed = 137


## Noise Model


In [ ]:
noise_config = SyntheticNoiseConfig(
    shots=2048,
    seed_simulator=seed,
    single_qubit_depolarizing=0.001,
    two_qubit_depolarizing=0.01,
    readout_error=0.02,
)

noisy_estimator = build_synthetic_noisy_aer_estimator(
    shots=noise_config.shots,
    seed_simulator=noise_config.seed_simulator,
    single_qubit_depolarizing=noise_config.single_qubit_depolarizing,
    two_qubit_depolarizing=noise_config.two_qubit_depolarizing,
    readout_error=noise_config.readout_error,
)

noise_config.metadata()


## H2 Systems and Grid

The grid mirrors the noiseless H2 grid: both bases, hardware-efficient ansatz choices, reps, and optimizers.


In [ ]:
grid_profile = "full"
systems = [
    system
    for system in statevector_grid_systems(profile=grid_profile)
    if system.name == "H2"
]

parameter_grid = {
    "mapper": ["jw"],
    "ansatz": ["real_amplitudes", "efficient_su2"],
    "reps": [1, 2],
    "optimizer": ["cobyla", "slsqp", "spsa"],
    "max_iter": [100],
    "seed": [seed],
}

total_jobs = sum(len(system.distances) for system in systems)
for values in parameter_grid.values():
    total_jobs *= len(values)

print(f"Systems: {len(systems)}")
print(f"Noisy H2 VQE jobs: {total_jobs}")
[(system.name, system.basis, len(system.distances), system.active_space) for system in systems]


## Execute Grid Search


In [ ]:
noisy_results_df = run_vqe_grid_search(
    systems=systems,
    parameter_grid=parameter_grid,
    estimator=noisy_estimator,
    cache=True,
    overwrite=False,
    include_fci_reference=True,
    run_label="noisy_synthetic",
    run_metadata=noise_config.metadata(),
)

noisy_results_df


### Summary


In [ ]:
summary_cols = [
    "run_label", "noise_source", "shots", "molecule", "basis", "distance",
    "mapper", "ansatz", "reps", "optimizer", "energy", "reference_energy",
    "abs_error_kcal_mol", "within_chemical_accuracy", "eval_count",
    "success", "error_type", "error",
]

noisy_results_df[[col for col in summary_cols if col in noisy_results_df.columns]].sort_values([
    "basis", "distance", "ansatz", "reps", "optimizer"
])


### Best Noisy Configuration by Point


In [ ]:
best_noisy_by_point = (
    noisy_results_df[noisy_results_df["success"].astype(bool)]
    .sort_values("abs_error_kcal_mol")
    .groupby(["molecule", "basis", "distance"], as_index=False)
    .head(1)
    .sort_values(["molecule", "basis", "distance"])
)

best_noisy_by_point[[col for col in summary_cols if col in best_noisy_by_point.columns]]


### Cache Inventory


In [ ]:
pd.DataFrame(summarize_data_dir()).sort_values(["molecule", "basis", "kind", "extension"])
